In [1]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))

True Tesla T4


In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer

dataset = load_dataset("stanfordnlp/imdb")
print(dataset)
print(dataset["train"][0]["text"][:200], dataset["train"][0]["label"])

name = "distilbert/distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(name)

def tokenize(batch):
    return tok(batch["text"], truncation=True, max_length=256)

train = dataset["train"].shuffle(seed=42).select(range(5000))
test = dataset["test"].shuffle(seed=42).select(range(1000))

train = train.map(tokenize, batched=True)
test = test.map(tokenize, batched=True)

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

... (7 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [3]:
!pip install -q evaluate

In [4]:
import numpy as np
import evaluate
from transformers import (AutoModelForSequenceClassification,
                          DataCollatorWithPadding,
                          TrainingArguments, Trainer)

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased",
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1},
)

accuracy = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

args = TrainingArguments(
    output_dir="/kaggle/working/out",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="no",
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train,
    eval_dataset=test,
    processing_class=tok,
    data_collator=DataCollatorWithPadding(tok),
    compute_metrics=compute_metrics,
)

trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.626959,0.867000
2,No log,0.588897,0.876000


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=314, training_loss=0.6202080599062002, metrics={'train_runtime': 135.2923, 'train_samples_per_second': 73.914, 'train_steps_per_second': 2.321, 'total_flos': 662336993280000.0, 'train_loss': 0.6202080599062002, 'epoch': 2.0})

In [5]:
from transformers import pipeline

clf = pipeline("sentiment-analysis", model=trainer.model, tokenizer=tok, device=0)

phrases = [
    "This movie was absolutely fantastic, I loved every minute!",
    "What a waste of time, the plot made no sense.",
    "It was okay, nothing special.",
    "I expected a masterpiece but got a boring mess.",
]
for p in phrases:
    print(p, "->", clf(p))

This movie was absolutely fantastic, I loved every minute! -> [{'label': 'POSITIVE', 'score': 0.9642711281776428}]
What a waste of time, the plot made no sense. -> [{'label': 'NEGATIVE', 'score': 0.9616939425468445}]
It was okay, nothing special. -> [{'label': 'NEGATIVE', 'score': 0.7076641917228699}]
I expected a masterpiece but got a boring mess. -> [{'label': 'NEGATIVE', 'score': 0.9648635983467102}]


In [6]:
#from kaggle_secrets import UserSecretsClient
#token = UserSecretsClient().get_secret("HF_TOKEN")

#from huggingface_hub import whoami
#user = whoami(token=token)["name"]
#repo_name = f"{user}/distilbert-imdb-sentiment"
#print("Dépôt :", repo_name)

#trainer.model.push_to_hub(repo_name, token=token)
#tok.push_to_hub(repo_name, token=token)